# 25 — Building a Custom MCP Server from Scratch

Every server you've called so far in this course -- the Azure MCP Server
(Parts 1-2) and whatever you exposed through API Management -- was written
for you. This notebook builds one from nothing, using the official MCP
Python SDK's high-level `FastMCP` API, so you understand what's on the
*other side* of every `ClientSession` you've used.

The server lives at
[examples/mcp_foundations/custom_server.py](../examples/mcp_foundations/custom_server.py)
-- open it alongside this notebook.

## `FastMCP`: the high-level server API

Three decorators cover almost everything a server needs to expose:

```python
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Foundations Demo Server")

@mcp.tool()
def add(a: int, b: int) -> int:
    '''Add two numbers together.'''
    return a + b

@mcp.resource("greeting://{name}")
def get_greeting(name: str) -> str:
    '''A parameterized resource, addressed by URI template.'''
    return f"Hello, {name}!"

@mcp.prompt()
def greet_user(name: str, style: str = "friendly") -> str:
    '''A reusable prompt template.'''
    return f"Write a {style} greeting for {name}."

if __name__ == "__main__":
    mcp.run()  # defaults to the stdio transport
```

`FastMCP` derives each tool's JSON Schema from its Python type hints and
docstring -- you never hand-write the schema the way a client sees it in
`tools/list`. Compare this with notebook 17: API Management's "expose a
REST API as MCP server" feature does the equivalent derivation from an
OpenAPI schema instead of Python type hints -- same destination (a tool
schema), different source.

In [ ]:
# Connect to our own server exactly the way the course has connected to the
# Azure MCP Server since notebook 03 -- same ClientSession/stdio_client
# pattern, just pointed at `python custom_server.py` instead of
# `npx @azure/mcp@latest`.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server_script = str(Path.cwd().parent / "examples" / "mcp_foundations" / "custom_server.py")
server_params = StdioServerParameters(command=sys.executable, args=[server_script])

async def list_everything():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await session.list_tools()
            resources = await session.list_resources()
            prompts = await session.list_prompts()
            return tools, resources, prompts

tools, resources, prompts = await list_everything()
print("tools:", [t.name for t in tools.tools])
print("resources:", [str(r.uri) for r in resources.resources])
print("prompts:", [p.name for p in prompts.prompts])

In [ ]:
# Call a tool on our own server -- identical shape to every
# azmcp_* tool call in Parts 1-2: call_tool(name, arguments) -> result.
async def call_add(a: int, b: int):
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            return await session.call_tool("add", arguments={"a": a, "b": b})

result = await call_add(5, 7)
print(result.content[0].text)

## Why build your own server at all?

- **Wrap an internal API** nobody's built an MCP server for yet -- expose
  it to GitHub Copilot or any MCP client without waiting for someone else
  to write that integration.
- **Prototype a tool's shape** before committing to a production
  implementation -- `FastMCP` lets you iterate on parameters/descriptions
  in minutes.
- **Expose a local dev tool** (a linter, a test runner, a scratch database)
  to an agent without any cloud dependency at all.

Once a server like this works locally over stdio, notebook 17 is how you'd
host it for real, governed clients: import it (or front it) through Azure
API Management and expose it as a remote Streamable HTTP MCP server.

## Exercise & next steps

Add a new `@mcp.tool()` to `custom_server.py` (for example, `multiply`),
restart the kernel, and re-run the cells above -- you should see it appear
in `tools/list` with no other changes needed.

Continue to
[26_mcp_resources_and_prompts.ipynb](26_mcp_resources_and_prompts.ipynb) to
exercise the two primitives this notebook only listed: resources and
prompts.